# StackOverflow JSONL Stats via scripts/report

This notebook reads the JSONL files produced by the StackOverflow conversion notebook and runs the same reporting utilities without recomputing XML parsing.

Use this after running [demo/00_stackoverflow_plain_to_lucene.ipynb](../demo/00_stackoverflow_plain_to_lucene.ipynb) so the JSONL collection files already exist.

In [10]:
from pathlib import Path
import shutil
import subprocess

REPO_ROOT = Path('/tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP')
COLLECT_ROOT = REPO_ROOT / 'demo' / 'collections'
SPLIT_NAME = 'test'
COLLECT_NAME = f'stackoverflow_{SPLIT_NAME}'
COLLECT_DIR = COLLECT_ROOT / COLLECT_NAME
SPLIT_DIR = COLLECT_DIR / 'input_data' / SPLIT_NAME
Q_JSONL = SPLIT_DIR / 'QuestionFields.jsonl'
A_JSONL = SPLIT_DIR / 'AnswerFields.jsonl'
QRELS = SPLIT_DIR / 'qrels.txt'
TMP_DIR = REPO_ROOT / 'scripts' / 'report' / 'tmp_stackoverflow_jsonl'
TMP_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT =', REPO_ROOT)
print('COLLECT_DIR =', COLLECT_DIR)
print('SPLIT_DIR   =', SPLIT_DIR)
print('Q_JSONL     =', Q_JSONL)
print('A_JSONL     =', A_JSONL)
print('QRELS       =', QRELS)
print('TMP_DIR     =', TMP_DIR)

REPO_ROOT = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP
COLLECT_DIR = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_test
SPLIT_DIR   = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_test/input_data/test
Q_JSONL     = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_test/input_data/test/QuestionFields.jsonl
A_JSONL     = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_test/input_data/test/AnswerFields.jsonl
QRELS       = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_test/input_data/test/qrels.txt
TMP_DIR     = /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/scripts/report/tmp_stackoverflow_jsonl


## 1) Confirm the converted JSONL exists

These files are created by the conversion notebook. This notebook only reads them.

In [12]:
for path in [Q_JSONL, A_JSONL, QRELS]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')

print('All required files exist.')

All required files exist.


## 2) Run token stats on the JSONL collection files

`scripts/report/print_field_tok_stat.py` expects JSONL, so this notebook reads the already generated collection JSONL directly.

In [13]:
def run_token_stats():
    cmd = f'''
set -e
cd {REPO_ROOT}

python scripts/report/print_field_tok_stat.py --input {Q_JSONL} --field text
python scripts/report/print_field_tok_stat.py --input {A_JSONL} --field text
'''
    subprocess.run(cmd, shell=True, check=True, text=True)

run_token_stats()

# of entries 73106 avg. tokens per entry: 48.36484009520422 total # of tokens: 3535760
# of entries 73106 avg. tokens per entry: 32.73491915848221 total # of tokens: 2393119


## 3) Qrels summary

The qrels file stays plain text, so this uses simple shell reporting.

In [16]:
cmd = f'''
set -e
cd {SPLIT_DIR}

echo 'line counts:'
wc -l QuestionFields.jsonl AnswerFields.jsonl qrels.txt

echo
echo 'qrels summary:'
awk '{{pairs++; q[$1]=1; d[$3]=1; g[$4]++}} END{{
  print "pairs="pairs, "queries="length(q), "docs="length(d);
  for (k in g) print "grade", k, "count", g[k];
}}' qrels.txt | sort -k2,2n
'''
subprocess.run(cmd, shell=True, check=True, text=True)

line counts:
    73106 QuestionFields.jsonl
    73106 AnswerFields.jsonl
    73106 qrels.txt
   219318 total

qrels summary:
pairs=73106 queries=73106 docs=73106
grade 1 count 73106


CompletedProcess(args='\nset -e\ncd /tempory/the_three_potatoes/ri_project/workspaces/amelie/FlexNeuART-IR-TTP/demo/collections/stackoverflow_test/input_data/test\n\necho \'line counts:\'\nwc -l QuestionFields.jsonl AnswerFields.jsonl qrels.txt\n\necho\necho \'qrels summary:\'\nawk \'{pairs++; q[$1]=1; d[$3]=1; g[$4]++} END{\n  print "pairs="pairs, "queries="length(q), "docs="length(d);\n  for (k in g) print "grade", k, "count", g[k];\n}\' qrels.txt | sort -k2,2n\n', returncode=0)

## Notes

- If you want another split, change `SPLIT_NAME` in Cell 2 and rerun from there.
- This notebook depends on the conversion notebook having already created the collection JSONL files.